# 03 · Early Prediction of Loss to Follow-up in Postpartum BP Monitoring

**Question.** At the end of postpartum day 3, using only what the platform can observe, can we identify which patients will stop sending blood pressure readings before day 14, so that outreach reaches them before the ACOG 7 to 10 day check and the BP peak have passed?

**Why it matters.** Remote monitoring works for patients who keep using it. If disengagement is concentrated among publicly insured, Black, or non-English-speaking patients, an early-warning model is how a program redirects scarce nurse outreach toward them. The same model can also *worsen* inequity if it is less accurate for those groups, so subgroup evaluation is part of the design and not an afterthought.

**Data.** Simulated cohort from `pp_synth.py` (n = 3,000). Every parameter is sourced or labeled as an assumption in that file.

> **Interpretation limit.** The outcome was generated from known rules, so this notebook validates the *pipeline and evaluation design*. It cannot establish how well such a model performs on real patients; that requires a real cohort (for example, a UCSF postpartum program or All of Us).

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import pp_synth as ps

SEED = 2026
PRED_DAY = 3      # predict at end of postpartum day 3
HORIZON = 14      # outcome: no readings on or after day 14

## 1 · Cohort, outcome, and features available at day 3

In [ ]:
params = ps.CohortParams(n_patients=3000, seed=SEED)
patients = ps.generate_patients(params)
readings = ps.generate_readings(patients, params)
summary = ps.engagement_summary(patients, readings)

# Outcome: lost to follow-up = no reading on or after day 14 (includes never-engaged).
y = (~summary.still_active_day14).astype(int).rename("lost_by_day14")

# Features: strictly limited to information available at the end of day PRED_DAY.
early = readings[readings.postpartum_day <= PRED_DAY]
g = early.groupby("patient_id")
feat = pd.DataFrame({
    "n_readings_d1_3": g.size(),
    "days_with_reading_d1_3": g.postpartum_day.nunique(),
    "max_sbp_d1_3": g.systolic_blood_pressure_value.max(),
    "mean_dbp_d1_3": g.diastolic_blood_pressure_value.mean(),
}).reindex(summary.index)
feat["any_reading_by_d3"] = feat.n_readings_d1_3.notna().astype(int)
feat[["n_readings_d1_3", "days_with_reading_d1_3"]] = feat[["n_readings_d1_3", "days_with_reading_d1_3"]].fillna(0)
feat[["max_sbp_d1_3", "mean_dbp_d1_3"]] = feat[["max_sbp_d1_3", "mean_dbp_d1_3"]].fillna(0)  # paired with indicator
X = feat.join(summary[["age", "insurance", "language", "hdp_subtype", "race_ethnicity"]])

display(Markdown(f"**{len(X):,} patients · outcome prevalence {y.mean():.0%} lost by day {HORIZON}**"))
display(pd.concat([X, y], axis=1).groupby("insurance").lost_by_day14.mean().to_frame("lost by day 14").style.format("{:.0%}"))

## 2 · Models

Two specifications, each fit as logistic regression and as gradient boosting:

- **Without race** (primary): engagement features, insurance, language, diagnosis, age.
- **With race**: adds race/ethnicity. Including race can improve calibration within groups but encodes a social category as a risk factor; the comparison makes that trade-off explicit for the team to decide.

A held-out test set (30%) is used once, after model selection by 5-fold cross-validation on the training set.

In [ ]:
NUM = ["n_readings_d1_3", "days_with_reading_d1_3", "max_sbp_d1_3", "mean_dbp_d1_3", "any_reading_by_d3", "age"]
CAT_BASE = ["insurance", "language", "hdp_subtype"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEED)

def make(model, cats):
    pre = ColumnTransformer([("num", StandardScaler(), NUM),
                             ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), cats)])
    return Pipeline([("pre", pre), ("clf", model)])

specs = {
    "Logistic, no race": (LogisticRegression(max_iter=2000), CAT_BASE),
    "Logistic, with race": (LogisticRegression(max_iter=2000), CAT_BASE + ["race_ethnicity"]),
    "Boosting, no race": (HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, random_state=SEED), CAT_BASE),
    "Boosting, with race": (HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, random_state=SEED), CAT_BASE + ["race_ethnicity"]),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
cv_rows, fitted = [], {}
for name, (m, cats) in specs.items():
    pipe = make(m, cats)
    auc = cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring="roc_auc")
    cv_rows.append({"model": name, "CV AUROC": f"{auc.mean():.3f} ± {auc.std():.3f}"})
    fitted[name] = pipe.fit(X_tr, y_tr)
display(pd.DataFrame(cv_rows).style.hide(axis="index"))

### Reference points

A model is only as good as what it is compared against. Two references bound the result:

- **Rule of thumb:** flag anyone with no reading by day 3 (what a nurse could do without a model).
- **Ceiling:** a model given the generator's hidden engagement trait, which no real system could observe. It shows how much signal exists in principle.

In [ ]:
def boot_auc(y_true, score, n=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    y_true, score = np.asarray(y_true), np.asarray(score)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(y_true), len(y_true))
        if y_true[i].min() != y_true[i].max():
            vals.append(roc_auc_score(y_true[i], score[i]))
    return np.percentile(vals, [2.5, 97.5])

rows = []
rule = 1 - X_te.any_reading_by_d3
lo, hi = boot_auc(y_te, rule)
rows.append(("Rule: no reading by day 3", roc_auc_score(y_te, rule), lo, hi, np.nan))

for name, pipe in fitted.items():
    p = pipe.predict_proba(X_te)[:, 1]
    lo, hi = boot_auc(y_te, p)
    rows.append((name, roc_auc_score(y_te, p), lo, hi, brier_score_loss(y_te, p)))

Xo = X.assign(trait=summary.engagement_trait_true)
oracle = make(LogisticRegression(max_iter=2000), CAT_BASE + ["race_ethnicity"])
oracle.named_steps["pre"].transformers[0] = ("num", StandardScaler(), NUM + ["trait"])
oracle.fit(Xo.loc[X_tr.index], y_tr)
p = oracle.predict_proba(Xo.loc[X_te.index])[:, 1]
lo, hi = boot_auc(y_te, p)
rows.append(("Ceiling (sees hidden trait)", roc_auc_score(y_te, p), lo, hi, brier_score_loss(y_te, p)))

results = pd.DataFrame(rows, columns=["model", "AUROC", "95% CI low", "95% CI high", "Brier"])
display(results.style.format({"AUROC": "{:.3f}", "95% CI low": "{:.3f}", "95% CI high": "{:.3f}", "Brier": "{:.3f}"}, na_rep="n/a").hide(axis="index"))

## 3 · Calibration on the test set

In [ ]:
PRIMARY = "Logistic, no race"
p_te = pd.Series(fitted[PRIMARY].predict_proba(X_te)[:, 1], index=X_te.index)
cal = pd.DataFrame({"p": p_te, "y": y_te})
cal["bin"] = pd.qcut(cal.p, 10, labels=False, duplicates="drop")
cb = cal.groupby("bin").agg(predicted=("p", "mean"), observed=("y", "mean"), n=("y", "size"))
fig = go.Figure([go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(dash="dash", color="#aaa"), name="perfect"),
                 go.Scatter(x=cb.predicted, y=cb.observed, mode="lines+markers", name=PRIMARY)])
fig.update_layout(title="Calibration by decile of predicted risk", xaxis_title="Predicted probability of loss",
                  yaxis_title="Observed proportion lost", height=420, xaxis_range=[0, 1], yaxis_range=[0, 1])
fig.show()

## 4 · Subgroup performance and outreach allocation

The operational question is concrete: a nurse team can call a fixed share of patients on day 4. If calls go to the highest-risk 30%, what share of each group's eventual dropouts does the team reach? A model with good overall AUROC can still miss a group systematically; this table is where that would show.

In [ ]:
CAPACITY = 0.30

def subgroup_table(name, pipe):
    p = pd.Series(pipe.predict_proba(X_te)[:, 1], index=X_te.index)
    flagged = p >= p.quantile(1 - CAPACITY)
    df = X_te.assign(y=y_te, p=p, flagged=flagged)
    out = []
    for by in ["insurance", "race_ethnicity", "language"]:
        for grp, s in df.groupby(by):
            out.append({
                "model": name, "attribute": by, "group": grp, "n": len(s),
                "observed loss": s.y.mean(),
                "mean predicted": s.p.mean(),
                "AUROC": roc_auc_score(s.y, s.p) if s.y.nunique() > 1 else np.nan,
                "share flagged": s.flagged.mean(),
                "dropouts reached": s.loc[s.y == 1, "flagged"].mean(),
            })
    return pd.DataFrame(out)

sub = pd.concat([subgroup_table(n, fitted[n]) for n in ["Logistic, no race", "Logistic, with race"]])
fmt = {c: "{:.0%}" for c in ["observed loss", "mean predicted", "share flagged", "dropouts reached"]} | {"AUROC": "{:.2f}"}
display(sub[sub.model == PRIMARY].drop(columns="model").style.format(fmt).hide(axis="index"))

In [ ]:
cmp = sub.pivot_table(index=["attribute", "group"], columns="model", values=["dropouts reached", "mean predicted"])
cmp.columns = [f"{a} · {m.replace('Logistic, ', '')}" for a, m in cmp.columns]
cmp["observed loss"] = sub[sub.model == PRIMARY].set_index(["attribute", "group"])["observed loss"]
display(Markdown("**Effect of including race as a predictor** (test set, 30% outreach capacity)"))
display(cmp.style.format("{:.0%}"))

In [ ]:
imp = sub[(sub.model == PRIMARY)]
fig = px.bar(imp, x="group", y="dropouts reached", facet_col="attribute", color="attribute",
             title=f"Share of each group's eventual dropouts reached by day-4 outreach (top {CAPACITY:.0%} by risk)",
             labels={"dropouts reached": "", "group": ""}, height=400)
fig.update_xaxes(matches=None).update_yaxes(tickformat=".0%", range=[0, 1]).update_layout(showlegend=False)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1].replace("_", " ")))
fig.show()

## 5 · What drives the primary model

In [ ]:
pipe = fitted[PRIMARY]
names = pipe.named_steps["pre"].get_feature_names_out()
coef = pd.Series(pipe.named_steps["clf"].coef_[0], index=[n.split("__")[1] for n in names])
coef = coef.reindex(coef.abs().sort_values().index)
px.bar(x=coef.values, y=coef.index, orientation="h", height=420,
       title="Logistic coefficients (standardized; positive = higher risk of loss)",
       labels={"x": "log-odds", "y": ""}).show()

In [ ]:
best = results.set_index("model")
r = best.loc[PRIMARY]
display(Markdown(f"""## Summary

- **Primary model** ({PRIMARY}): test AUROC **{r.AUROC:.2f}** (95% CI {r['95% CI low']:.2f} to {r['95% CI high']:.2f}), Brier {r.Brier:.3f}.
- **Rule of thumb** (no reading by day 3): AUROC {best.loc['Rule: no reading by day 3', 'AUROC']:.2f}.
- **Ceiling** with the hidden engagement trait: AUROC {best.loc['Ceiling (sees hidden trait)', 'AUROC']:.2f}. The gap between the primary model and the ceiling is the signal a richer engagement feed (reminder responses, app opens, wearable wear time) could recover.
- Subgroup results above show whether day-4 outreach reaches each group's dropouts in proportion to need.

These results describe a simulated cohort and validate the pipeline and evaluation design. Performance on real patients is unknown until the model is trained and tested on real postpartum monitoring data."""))